<a href="https://colab.research.google.com/github/dhiru69-tech/ML-Techniques-Lab-Project/blob/main/notebooks/day08_evaluation_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Cell 1: Setup (same as Day 7) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Load data. bad = 1 is the positive class we want to catch
data = fetch_openml(name="credit-g", version=1, as_frame=True)
df = data.frame
X = df.drop(columns=['class'])
y = df['class'].map({'good': 0, 'bad': 1}).astype(int)

# Stratified 80/20 split with a fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numeric columns and one-hot encode categorical columns (fit on train only)
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

preprocess = ColumnTransformer(
    [('num', StandardScaler(), num_cols),
     ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)],
    sparse_threshold=0
)
preprocess.set_output(transform="pandas")

X_train_p = preprocess.fit_transform(X_train)
X_test_p = preprocess.transform(X_test)

print("Train:", X_train_p.shape, "| Test:", X_test_p.shape)
print("Bad customers in test set:", y_test.sum(), "out of", len(y_test))

Train: (800, 61) | Test: (200, 61)
Bad customers in test set: 60 out of 200


In [2]:
# --- Cell 2: Baseline ("always good") and its confusion matrix ---
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix

baseline = DummyClassifier(strategy='most_frequent')   # always predicts the majority class
baseline.fit(X_train_p, y_train)
pred_base = baseline.predict(X_test_p)

cm = confusion_matrix(y_test, pred_base)
print(cm)

# ravel() flattens the 2x2 table in this order: TN, FP, FN, TP
tn, fp, fn, tp = cm.ravel()
print("TN =", tn, "| FP =", fp, "| FN =", fn, "| TP =", tp)

[[140   0]
 [ 60   0]]
TN = 140 | FP = 0 | FN = 60 | TP = 0
